# Impact of Generative AI Usage on Academic Performance

## Objective

Analyze whether the use of Generative AI tools has a significant relationship with university students' academic performance.


## Research Questions

1. Is there a relationship between Generative AI usage and final GPA?
2. Do AI-related variables improve predictive model performance?
3. Which factors best explain academic performance?


# Research Hypotheses

H0:
Generative AI usage has no significant relationship with academic performance.

H1:
Generative AI usage has a significant relationship with academic performance.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/ai_student_impact_dataset.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'ai_student_impact_dataset.csv'

# Data Understanding

## Data Preparation

This section focuses on data validation, variable inspection, and preprocessing steps required for subsequent analyses.

In [ ]:
df.head()
#Observación de las 5 filas primeras para entender a simple vista el conjunto de datos.

In [ ]:
df.dtypes
# Observación de las columnas y su tipo de dato.

In [ ]:
df.describe()
#Descripción de variables numéricas para observar sus tendencias.

In [ ]:
#Creación de variables que separen las columnas categóricas por un lado y por otro lado las columnas numéricas
cat_cols = ["Major_Category","Year_of_Study","Primary_Use_Case","Prompt_Engineering_Skill","Institutional_Policy","Burnout_Risk_Level"]
num_cols = ["Pre_Semester_GPA","Post_Semester_GPA","Weekly_GenAI_Hours","Traditional_Study_Hours","Perceived_AI_Dependency","Anxiety_Level_During_Exams","Skill_Retention_Score"]

# Feature Engineering

## GPA Change Variable

A new variable was created to measure changes in academic performance between semesters. This metric provides additional insight into how different factors may influence student outcomes over time.

In [ ]:
df["GPA_Change"] = (
    df["Post_Semester_GPA"]
    - df["Pre_Semester_GPA"]
)

# Exploratory Data Analysis (EDA)







### Distribution of Numerical Variables


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(18,10))

variables = [
    'Pre_Semester_GPA',
    'Post_Semester_GPA',
    'Weekly_GenAI_Hours',
    'Traditional_Study_Hours',
    'Perceived_AI_Dependency',
    "Skill_Retention_Score"
]

for ax, var in zip(axes.flatten(), variables):
    sns.histplot(df[var], kde=True, ax=ax)
    ax.set_title(f'Distribution of {var}')

plt.tight_layout()
plt.show()

### Initial GPA vs Final GPA



In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=df,
    x='Pre_Semester_GPA',
    y='Post_Semester_GPA'
)

plt.plot(
    [df['Pre_Semester_GPA'].min(), df['Pre_Semester_GPA'].max()],
    [df['Pre_Semester_GPA'].min(), df['Pre_Semester_GPA'].max()],
    'r--'
)

plt.title('Initial GPA vs Final GPA')
plt.xlabel('Initial GPA')
plt.ylabel('Final GPA')
plt.show()

### Correlation Analysis

In [ ]:
matriz_correlacion = df[num_cols].corr()

plt.figure(figsize=(10, 8))
# 3. Creamos el Mapa de Calor
# annot=True imprime el número exacto
# cmap='coolwarm' usa azul para correlaciones negativas y rojo para positivas
sns.heatmap(
    matriz_correlacion,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5,
    vmin=-1, vmax=1
)

plt.title('Heatmap: Correlation between variables', fontsize=16, pad=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### AI Usage and Academic Performance


In [ ]:
plt.figure(figsize=(8,6))

sns.regplot(
    data=df,
    x='Weekly_GenAI_Hours',
    y='Post_Semester_GPA',
    scatter_kws={'alpha':0.6}
)

plt.title('AI Usage vs Final GPA')
plt.xlabel('AI Usage')
plt.ylabel('Final GPA')
plt.show()

### Study Hours and Academic Performance




In [ ]:
plt.figure(figsize=(8,6))

sns.regplot(
    data=df,
    x='Traditional_Study_Hours',
    y='Post_Semester_GPA',
    scatter_kws={'alpha':0.5},
    line_kws={'color':'red'}
)

plt.title('Study hours vs Final GPA')
plt.xlabel('Study hours')
plt.ylabel('Final GPA')

plt.show()

### Performance Change Statistics




In [ ]:
#Observación de estadísticos del cambio de GPA.
df['GPA_Change'].describe()

### GPA Change Distribution


In [ ]:
plt.figure(figsize=(8,6))

sns.histplot(
    df['GPA_Change'],
    bins=20,
    kde=True
)

plt.axvline(
    df['GPA_Change'].mean(),
    color='red',
    linestyle='--',
    label='Promedio'
)

plt.legend()
plt.title('GPA change distribution')

plt.show()

### AI Usage and GPA Change


In [ ]:
plt.figure(figsize=(8,6))

sns.regplot(
    data=df,
    x='Weekly_GenAI_Hours',
    y='GPA_Change'
)

plt.title('AI Usage vs GPA change')
plt.xlabel('AI usage')
plt.ylabel('GPA change')

plt.show()

### GPA Change by AI Usage Intensity

---



In [ ]:
#Gráfico para observar el cambio de GPA según diferentes intensidades en el uso de IA generativa.

df['AI_Group'] = pd.cut(
    df['Weekly_GenAI_Hours'],
    bins=[0,5,10,15,20],
    labels=[
        'Low',
        'Moderate',
        'High',
        'Very high'
    ]
)

plt.figure(figsize=(8,6))

sns.boxplot(
    data=df,
    x='AI_Group',
    y='GPA_Change'
)

plt.title('GPA change according to AI intensity usage')
plt.xlabel('Intensity of AI usage')
plt.ylabel('GPA change')

plt.show()

# Hierarchical Regression Analysis

To evaluate the incremental contribution of AI-related variables, three hierarchical linear regression models were developed with increasing levels of complexity.

Model 1:
Traditional academic predictors.

Model 2:
Traditional predictors plus weekly AI usage.

Model 3:
Traditional predictors plus advanced AI-related variables.



In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

#Transformación de las variables categóricas en variables numéricas
df_procesado = pd.get_dummies(df, columns=cat_cols, drop_first=True)

#Variable objetivo de nuestro modelo
target = 'Post_Semester_GPA'


In [ ]:
# Definimos los bloques de variables (La escalera analítica)
bloque_1_base = ['Pre_Semester_GPA', 'Traditional_Study_Hours'] #Modelo base
bloque_2_tiempo_ia = ['Weekly_GenAI_Hours']

# Capturamos todas las variables dummy de habilidades y casos de uso
bloque_3_habilidades_ia = [col for col in df_procesado.columns if 'Primary_Use_Case_' in col or 'Prompt_Engineering_Skill_' in col or "Skill_Retention_Score" in col or "Paid_Subscription" in col]

# Limpiamos nulos para que todos los modelos evalúen exactamente a los mismos alumnos
columnas_totales = bloque_1_base + bloque_2_tiempo_ia + bloque_3_habilidades_ia
df_procesado = df_procesado.dropna(subset=columnas_totales + [target])

# Dividimos datos de entrenamiento y prueba (80/20)
df_train, df_test = train_test_split(df_procesado, test_size=0.2, random_state=42)
y_train = df_train[target]
y_test = df_test[target]

# Función para entrenar y devolver el R²
def evaluar_paso(variables):
    modelo = LinearRegression()
    modelo.fit(df_train[variables], y_train)
    preds = modelo.predict(df_test[variables])
    return r2_score(y_test, preds)

# ==========================================
# EJECUCIÓN PASO A PASO
# ==========================================
print("--- ANÁLISIS DE REGRESIÓN JERÁRQUICA ---\n")

# PASO 1: El Estudiante Tradicional
vars_paso_1 = bloque_1_base
r2_paso_1 = evaluar_paso(vars_paso_1)
print(f"PASO 1 (Modelo Base): Historial + Estudio Tradicional")
print(f"R² alcanzado: {r2_paso_1:.4f}\n")

# PASO 2: Adopción de IA (Solo el tiempo de uso)
vars_paso_2 = vars_paso_1 + bloque_2_tiempo_ia
r2_paso_2 = evaluar_paso(vars_paso_2)
delta_2 = r2_paso_2 - r2_paso_1
print(f"PASO 2 (+ Tiempo de IA): Agregamos 'Weekly_GenAI_Hours'")
print(f"R² alcanzado: {r2_paso_2:.4f}")
print(f"Mejora (Delta R²): {delta_2:.4f} (Aporta un {(delta_2 * 100):.2f}% extra)\n")

# PASO 3: Adopción Sofisticada (Cómo y qué tan bien la usa)
vars_paso_3 = vars_paso_2 + bloque_3_habilidades_ia
r2_paso_3 = evaluar_paso(vars_paso_3)
delta_3 = r2_paso_3 - r2_paso_2
print(f"PASO 3 (+ Habilidad y Casos de Uso): Agregamos Prompts y Tipo de Uso")
print(f"R² alcanzado: {r2_paso_3:.4f}")
print(f"Mejora (Delta R²): {delta_3:.4f} (Aporta un {(delta_3 * 100):.2f}% extra)\n")

# Resumen Final
impacto_total_ia = r2_paso_3 - r2_paso_1
print(f"--- CONCLUSIÓN ---")
print(f"El impacto neto total de TODAS las variables de IA sumadas es de {(impacto_total_ia * 100):.2f}% en la nota final.")

# Results Interpretation

The baseline model, which included traditional academic predictors such as previous GPA and study hours, explained 87.89% of the variance in final GPA.

Adding Weekly_GenAI_Hours did not result in any meaningful improvement in model performance (ΔR² = 0.0000).

However, when more sophisticated AI-related variables, including prompt quality and AI use cases, were introduced, the model's R² increased to 0.8873.

Although statistically measurable, this increase corresponds to only a 0.85 percentage point improvement in explanatory power.

Overall, the results indicate that Generative AI usage contributes some additional information for understanding academic performance, but its predictive value remains substantially lower than that of established academic factors such as prior achievement and study habits.

# Statistical Inference

While the previous section focused on the predictive contribution of AI-related variables, this section examines their statistical significance.

Specifically, the analysis evaluates whether variables associated with Generative AI usage exert a significant effect on final GPA after accounting for established academic predictors.

## Statistical Hypotheses

H0:
AI-related variables have no significant effect on final GPA.

H1:
At least one AI-related variable has a significant effect on final GPA.

In [ ]:
#MODELO BASE: solo con variables de GPA pre semestre y horas de estudio

import statsmodels.api as sm

X_base = df[
    [
        'Pre_Semester_GPA',
        'Traditional_Study_Hours'
    ]
]

X_base = sm.add_constant(X_base)

y = df['Post_Semester_GPA']

model_base = sm.OLS(y, X_base).fit()

print(model_base.summary())

In [ ]:
#MODELO COMPLEJO: Se agregan variables: tiempo de uso en IA, diversidad de herramientas, habilidad de retención.

X_ai = df[
    [
        'Pre_Semester_GPA',
        'Traditional_Study_Hours',
        'Weekly_GenAI_Hours',
        "Prompt_Engineering_Skill",
        "Skill_Retention_Score",
        "Primary_Use_Case",
        "Paid_Subscription"
    ]
]

X_ai = pd.get_dummies(
    X_ai,
    drop_first=True
)

bool_cols = X_ai.select_dtypes(include='bool').columns

X_ai[bool_cols] = X_ai[bool_cols].astype(int)

X_ai = sm.add_constant(X_ai)

model_ai = sm.OLS(y, X_ai).fit()

print(model_ai.summary())

In [2]:
comparison = pd.DataFrame({
    'Modelo': ['Base', 'Con IA'],
    'R2': [
        model_base.rsquared,
        model_ai.rsquared
    ]
})

comparison

NameError: name 'model_base' is not defined

In [ ]:
delta_r2 = (
    model_ai.rsquared
    - model_base.rsquared
)

print(
    f"Incremento de R² por IA: {delta_r2:.4f}"
)

In [ ]:
conf = model_ai.conf_int()

conf.columns = [
    'Limite Inferior',
    'Limite Superior'
]

conf

In [ ]:
coef_table = pd.DataFrame({
    'Coeficiente': model_ai.params,
    'P_Value': model_ai.pvalues
})

coef_table

In [ ]:
significant = coef_table[
    coef_table['P_Value'] < 0.05
]

significant

In [ ]:
coef_plot = coef_table.drop('const')

plt.figure(figsize=(8,6))

sns.barplot(
    x=coef_plot['Coeficiente'],
    y=coef_plot.index
)

plt.title(
    'Model coefficients'
)
plt.xlabel('Coef value')
plt.ylabel('Variable')

plt.show()

## Conclusions

The results confirm that previous GPA remains the strongest predictor of future academic performance.

Traditional study hours also maintain a positive and statistically significant relationship with final GPA, reinforcing the importance of established academic habits.

Regarding AI-related variables, weekly usage hours exhibit a statistically significant but relatively small effect size, suggesting a limited practical contribution to academic outcomes.

In contrast, variables associated with the quality of AI usage show more meaningful effects. Students with stronger prompt engineering skills and those who use AI for problem-solving activities tend to achieve better academic results.

Conversely, relying on AI primarily for direct answer generation is associated with lower academic performance.

Overall, the findings indicate that the characteristics of AI usage are more strongly associated with academic performance than the amount of time spent using AI tools.


# Random Forest Analysis

Following the statistical inference analysis, a Random Forest model was implemented to explore potentially non-linear relationships among variables and assess their relative importance in predicting academic performance.

Unlike linear regression models, Random Forest can capture complex interactions and non-linear patterns that may not be detected through traditional statistical approaches. Additionally, the model provides feature importance scores, allowing for the identification of the variables that contribute most to prediction accuracy.


In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

X = df_procesado[columnas_totales]

y = df['Post_Semester_GPA']

X = pd.get_dummies(
    X,
    drop_first=True
)

X_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

rf_modelo = RandomForestRegressor(n_estimators=100,max_depth=10, random_state=42)
rf_modelo.fit(X_train_comp, y_train_comp)

rf_predicciones = rf_modelo.predict(X_test_comp)
rf_r2 = r2_score(y_test_comp, rf_predicciones)
rf_mse = mean_squared_error(y_test_comp, rf_predicciones)
rf_mae = mean_absolute_error(y_test_comp, rf_predicciones)

print("--- Evaluación del Random Forest ---")
print(f"R²:   {rf_r2:.4f}")
print(f"RMSE: {rf_mse**0.5:.4f}\n")
print(f"MAE:  {rf_mae:.4f}\n")

NameError: name 'df_procesado' is not defined

In [ ]:
importances = rf_modelo.feature_importances_
nombres_caracteristicas = X_train_comp.columns

df_importancia = pd.DataFrame({ 'Característica': nombres_caracteristicas, 'Importancia': importances })
df_importancia = df_importancia.sort_values(by='Importancia', ascending=False)

plt.figure(figsize=(10, 6))

sns.barplot(data=df_importancia, x='Importancia', y='Característica', palette='magma')
plt.title('Importance of characteristic on Random Forest model')
plt.xlabel('Importance')
plt.ylabel('Characteristic')
plt.tight_layout()
plt.show()


In [ ]:
X_sin_gpa = X.drop(
    columns=['Pre_Semester_GPA']
)

X_train_sp, X_test_sp, y_train_sp, y_test_sp = train_test_split(
    X_sin_gpa,
    y,
    test_size=0.2,
    random_state=42
)

rf_modelo_sp = RandomForestRegressor(n_estimators=100,max_depth=10, random_state=42)
rf_modelo_sp.fit(X_train_sp, y_train_sp)

predicciones_sin_gpa = rf_modelo_sp.predict(X_test_sp)
r2_sin_gpa = r2_score(y_test_sp, predicciones_sin_gpa)
mse_sin_gpa = mean_squared_error(y_test_sp, predicciones_sin_gpa)
mae_sin_gpa = mean_absolute_error(y_test_sp, predicciones_sin_gpa)

print("--- Evaluación del Random Forest sin GPA ---")
print(f"R²:   {r2_sin_gpa:.4f}")
print(f"RMSE: {mse_sin_gpa**0.5:.4f}\n")
print(f"MAE:  {mae_sin_gpa:.4f}\n")

## Random Forest Model Evaluation

The Random Forest model achieved an R² of 0.8970, explaining approximately 89.7% of the variance in final GPA.

Compared with the best hierarchical regression model, the performance gain was marginal. This suggests that non-linear patterns contribute only limited additional explanatory power beyond what is already captured by traditional linear models.

The consistency between both approaches strengthens confidence in the overall findings and indicates that the primary relationships between academic performance and the predictor variables are largely linear in nature.

## Additional Analysis

To evaluate the contribution of variables beyond academic history, a second Random Forest model was trained after excluding the previous GPA variable.

The model achieved an R² of 0.053, indicating that the remaining variables explain only 5.3% of the observed variability in final GPA.

This result reinforces the evidence obtained in the previous analyses: prior academic performance is by far the strongest predictor of future achievement, whereas AI-related variables possess substantially lower explanatory power.

The sharp decline in model performance after removing previous GPA highlights the dominant role of academic history in predicting student outcomes.


# Final Conclusions

This study examined the relationship between Generative AI usage and academic performance among university students using exploratory analysis, statistical inference, hierarchical regression, and machine learning techniques.

Across all analyses, previous GPA consistently emerged as the strongest predictor of final academic performance. Traditional academic factors, particularly prior achievement and study habits, explained the vast majority of the observed variability in student outcomes.

The results indicate that Generative AI usage is associated with academic performance; however, its contribution is considerably smaller than that of traditional academic variables. Weekly AI usage hours showed a statistically significant but limited practical effect, while variables related to the quality and purpose of AI usage demonstrated stronger associations with academic outcomes.

Students who reported stronger prompt engineering skills and those who used AI as a problem-solving tool tended to achieve better academic results. In contrast, using AI primarily for direct answer generation was associated with lower academic performance.

The Random Forest analysis largely confirmed the findings obtained through linear modeling. The minimal performance improvement over hierarchical regression suggests that the relationships between the analyzed variables and academic performance are predominantly linear. Furthermore, removing previous GPA from the model resulted in a substantial decline in predictive performance, reinforcing the dominant role of academic history.

Overall, the findings suggest that the way students use Generative AI is more strongly associated with academic performance than the amount of time spent using these tools. While AI-related variables provide additional explanatory information, academic success remains primarily driven by established educational factors.


# Limitations

This study has several limitations that should be considered when interpreting the results.

First, the data is observational, meaning the findings reflect associations rather than causal relationships.

Second, several variables were self-reported, which may introduce measurement bias due to perception or reporting inaccuracies.

Finally, the dataset does not include potentially relevant factors such as student motivation, socioeconomic context, or differences in educational quality. These unobserved variables may also influence academic performance and partially explain the results.